Paper link: https://arxiv.org/pdf/2102.06171

Key Terms:

Problem: Batch norm has many problems:
- computationally expensive
- introduces discrepancy between training and inference
- batch size dependent

Remind me what Batch Norm is: downscales the residual branch
- reduces the F(x) at initialization to be close to identity mapping, so that training becomes stable.

- reduces mean shift, which is the change in the distribution of layer inputs during training, as the parameters of the previous layers change. We don't want the deep networks to predict the same label for all inputs.

- regularizer - helps with optimization

Solution Architecture:

AGC applied to every layer except the last linear layer.

Mixup, Cutmix, RandAugment used for data augmentation.

Scaled Weight Standardization (Scaled W) applied to all convolutional layers.

- Scaled Weight Standardization normalizes the weights of convolutional layers by their standard deviation and scales them by a learnable parameter. This helps stabilize training by ensuring that the weights have a consistent scale, which can improve convergence and performance.

This method makes sure that each layer never produce unstable outputs.

- Think of each convolution filter as a “loudspeaker.”
- Without control: some layers shout, some whisper
- BatchNorm listens and adjusts volume dynamically
- Scaled WS builds every speaker to output at the same volume



In [6]:
import torch
from torch import nn
import torch.nn.functional as F
from torchvision import transforms, datasets
from torch.utils.data import DataLoader, Subset
import time

torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision('medium')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
torch.cuda.empty_cache()

# ─── Data ────────────────────────────────────────
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.5071, 0.4865, 0.4409), (0.2673, 0.2564, 0.2761)),
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761)),
])

cifar_train_raw = datasets.CIFAR100("./data", train=True, download=True)
train_size = int(0.9 * len(cifar_train_raw))
train_idx = list(range(train_size))
val_idx   = list(range(train_size, len(cifar_train_raw)))

train_ds = Subset(datasets.CIFAR100("./data", train=True, transform=train_transform), train_idx)
val_ds   = Subset(datasets.CIFAR100("./data", train=True, transform=test_transform),   val_idx)

batch_size = 256  # Smaller to reduce chance of bad statistics
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train batches: {len(train_loader)} | Samples: {len(train_ds)}")

# ─── Safer Scaled Conv ────────────────────────────

class ScaledStdConv2d(nn.Conv2d):
    def __init__(self, *args, bias=False, **kwargs):
        super().__init__(*args, bias=bias, **kwargs)
        # Very conservative init to prevent explosion
        fan_in = self.weight.size(1) * self.kernel_size[0] * self.kernel_size[1] // self.groups
        nn.init.normal_(self.weight, mean=0.0, std=0.05 / (fan_in ** 0.5))  # tiny variance
    
    def forward(self, x):
        w = self.weight
        mean = w.mean(dim=[1,2,3], keepdim=True)
        std = w.std(dim=[1,2,3], keepdim=True) + 1e-3  # larger eps
        w = (w - mean) / std * 0.5                     # very low scaling factor
        out = F.conv2d(x, w, None, self.stride, self.padding, self.dilation, self.groups)
        
        # Debug: catch NaN early
        if torch.isnan(out).any() or torch.isinf(out).any():
            print("NaN/Inf detected in conv output!")
        
        return out

class ResidualBlock(nn.Module):
    def __init__(self, in_c, out_c, stride=1):
        super().__init__()
        self.conv1 = ScaledStdConv2d(in_c,  out_c, 3, stride, 1)
        self.conv2 = ScaledStdConv2d(out_c, out_c, 3, 1,     1)
        self.act   = nn.ReLU(inplace=True)  # ReLU + inplace for speed/memory
        self.skip  = ScaledStdConv2d(in_c, out_c, 1, stride, 0, bias=False) if stride != 1 or in_c != out_c else nn.Identity()
    
    def forward(self, x):
        res = self.skip(x)
        y = self.act(self.conv1(x))
        y = self.conv2(y)
        out = y + res
        
        if torch.isnan(out).any() or torch.isinf(out).any():
            print("NaN/Inf in residual block output!")
        
        return out

class FastMiniNet(nn.Module):
    def __init__(self, num_classes=100):
        super().__init__()
        self.stem = nn.Sequential(
            ScaledStdConv2d(3,  32, 3, 1, 1), nn.ReLU(inplace=True),
            ScaledStdConv2d(32, 64, 3, 2, 1), nn.ReLU(inplace=True),
        )
        
        self.body = nn.Sequential(
            ResidualBlock(64,  96, stride=2),
            ResidualBlock(96,  128),
            ResidualBlock(128, 192, stride=2),
            ResidualBlock(192, 256),
        )
        
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc   = nn.Linear(256, num_classes)
        
        # Final layer init - small scale
        nn.init.normal_(self.fc.weight, std=0.01)
    
    def forward(self, x):
        x = self.stem(x)
        x = self.body(x)
        x = self.pool(x).flatten(1)
        out = self.fc(x)
        
        if torch.isnan(out).any() or torch.isinf(out).any():
            print("NaN/Inf in final output!")
        
        return out

model = FastMiniNet().to(device)
# Comment out torch.compile temporarily - it can sometimes mask/hide NaN issues
# model = torch.compile(model)

print(f"Params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

# ─── Very gentle optimizer setup ─────────────────
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-5)  # low fixed LR
loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)

# No AMP / GradScaler for now - full float32 to debug stability

# ─── Training Loop ───────────────────────────────
for epoch in range(40):
    start = time.time()
    model.train()
    train_loss = 0.0
    num_batches = 0

    for i, (x, y) in enumerate(train_loader):
        x, y = x.to(device), y.to(device)
        
        optimizer.zero_grad(set_to_none=True)
        
        out = model(x)
        loss = loss_fn(out, y)
        
        if torch.isnan(loss) or torch.isinf(loss):
            print(f"NaN/Inf loss at batch {i} (epoch {epoch+1}) — skipping")
            continue
        
        loss.backward()
        
        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        
        train_loss += loss.item()
        num_batches += 1
        
        if i % 50 == 0:
            print(f"Epoch {epoch+1} | Batch {i:3d} | loss {loss.item():.4f} | grad_norm {grad_norm:.3f}")
    
    avg_loss = train_loss / num_batches if num_batches > 0 else float('nan')
    
    print(f"Epoch {epoch+1:2d} | Train loss: {avg_loss:.4f} | Batches: {num_batches} | Time: {time.time()-start:.1f}s")

torch.cuda.empty_cache()

Device: cuda
Train batches: 176 | Samples: 45000
Params: 2,118,596
Epoch 1 | Batch   0 | loss 2547398.0000 | grad_norm 347674720.000
Epoch 1 | Batch  50 | loss 19651.6484 | grad_norm 15498751.000
Epoch 1 | Batch 100 | loss 802.9671 | grad_norm 269940.062
Epoch 1 | Batch 150 | loss 265.5287 | grad_norm 21976.531
Epoch  1 | Train loss: 22204947752.2125 | Batches: 176 | Time: 22.5s
Epoch 2 | Batch   0 | loss 77.6175 | grad_norm 146572.297
Epoch 2 | Batch  50 | loss 165.8503 | grad_norm 27127.945
Epoch 2 | Batch 100 | loss 81.1407 | grad_norm 11434.610
Epoch 2 | Batch 150 | loss 13.3898 | grad_norm 6811.442
Epoch  2 | Train loss: 124572.7265 | Batches: 176 | Time: 18.3s
Epoch 3 | Batch   0 | loss 12867.6094 | grad_norm 16427699.000
Epoch 3 | Batch  50 | loss 2042.9248 | grad_norm 26768954.000
Epoch 3 | Batch 100 | loss 38.1913 | grad_norm 74943.898
Epoch 3 | Batch 150 | loss 72.2602 | grad_norm 159189.250
Epoch  3 | Train loss: 28298.4923 | Batches: 176 | Time: 18.9s
Epoch 4 | Batch   0 | 

KeyboardInterrupt: 

In [ ]:
def evaluate_test_set(model):
    model.eval()
    correct = 0
    total = 0
    
    print("Starting evaluation...")
    
    with torch.no_grad():
        for data in test_loader:
            images, labels = data
            images, labels = images.to(device), labels.to(device)
            
            # Use autocast for consistency if you trained with it
            outputs = model(images)
            
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f'Accuracy of the network on the test images: {accuracy:.2f}%')
    return accuracy

print("\n=== Running standard evaluation ===")
standard_accuracy = evaluate_test_set(model)   
print(f'Standard Test Accuracy: {standard_accuracy:.4f} ({standard_accuracy*100:.2f}%)')